# Notebook 1. Python Data Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/D1lxrry/Nortstar-Task/blob/main/colab_notebooks/01_python_data_processing.ipynb)

The pandas side of the NorthStar Urban Mobility & Logistics coursework, end to end. 
Every cell pulls data from the GitHub repo so a fresh Colab session needs 
no local setup. Press **Runtime > Run all**.

**Sections**
1. Setup
2. Extract (9 CSVs from the repo)
3. Pre-clean data quality scorecard
4. Transform (zone canonicalisation, dates)
5. Post-clean scorecard
6. Build the analytical wide table
7. Descriptive statistics
8. scipy chi square tests
9. scikit-learn random forest
10. Rubric coverage table

## 1. Setup

In [ ]:
# Colab has pandas, numpy, scipy, sklearn, matplotlib already.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report
)
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
RAW = 'https://raw.githubusercontent.com/D1lxrry/Nortstar-Task/main'
print('setup done')

## 2. Extract: pull the 9 CSVs from the repo

Dataset lives at `northstar_dataset/` on the repo. I read each CSV via 
`raw.githubusercontent.com` so the marker does not need to upload anything.

In [ ]:
CSV_FILES = ['customers','orders','deliveries','drivers','vehicles',
             'hubs','incidents','complaints','app_events']
frames = {n: pd.read_csv(f'{RAW}/northstar_dataset/{n}.csv') for n in CSV_FILES}
for name, df in frames.items():
    print(f'{name:<12} {df.shape[0]:>5} rows  {df.shape[1]:>2} cols')

## 3. Pre-clean data quality scorecard

5 dimensions per table: row count, total NA cells, duplicate primary keys, 
foreign-key violations against the referenced PKs, and the distinct zone 
count across every zone-bearing column. Runs once before cleaning, once 
after, so the diff is visible.

In [ ]:
SCHEMA = {
    'customers':  {'pk': 'customer_id', 'fks': {},
                   'zone_cols': ['home_zone']},
    'orders':     {'pk': 'order_id',
                   'fks': {'customer_id': ('customers', 'customer_id')},
                   'zone_cols': ['pickup_zone', 'dropoff_zone']},
    'deliveries': {'pk': 'delivery_id',
                   'fks': {'order_id': ('orders', 'order_id'),
                           'driver_id': ('drivers', 'driver_id'),
                           'vehicle_id': ('vehicles', 'vehicle_id'),
                           'hub_id': ('hubs', 'hub_id')},
                   'zone_cols': []},
    'drivers':    {'pk': 'driver_id', 'fks': {}, 'zone_cols': ['base_zone']},
    'vehicles':   {'pk': 'vehicle_id', 'fks': {}, 'zone_cols': ['assigned_zone']},
    'hubs':       {'pk': 'hub_id', 'fks': {}, 'zone_cols': ['zone']},
    'incidents':  {'pk': 'incident_id',
                   'fks': {'delivery_id': ('deliveries', 'delivery_id')},
                   'zone_cols': []},
    'complaints': {'pk': 'complaint_id',
                   'fks': {'order_id': ('orders', 'order_id'),
                           'customer_id': ('customers', 'customer_id')},
                   'zone_cols': []},
    'app_events': {'pk': 'event_id',
                   'fks': {'customer_id': ('customers', 'customer_id'),
                           'order_id': ('orders', 'order_id')},
                   'zone_cols': ['zone_context']},
}

def scorecard(frames, schema, label):
    rows = []
    for name, df in frames.items():
        s = schema[name]
        pk = s['pk']
        nulls = int(df.isna().sum().sum())
        dup_pk = int(df[pk].duplicated().sum())
        fk_violations = 0
        for fk_col, (ref_table, ref_col) in s.get('fks', {}).items():
            valid = set(frames[ref_table][ref_col].dropna())
            fk_violations += int((~df[fk_col].isin(valid) & df[fk_col].notna()).sum())
        distinct_zones = int(sum(df[c].nunique() for c in s.get('zone_cols', [])))
        rows.append({'table': name, 'rows': len(df), 'nulls': nulls,
                     'dup_pk': dup_pk, 'fk_violations': fk_violations,
                     'distinct_zones': distinct_zones})
    out = pd.DataFrame(rows).set_index('table')
    print(f'\n{label}')
    print('-' * 60)
    print(out)
    return out

pre = scorecard(frames, SCHEMA, 'Pre-clean scorecard')

## 4. Transform: zone canonicalisation, dates

The pre-clean scorecard surfaces 16 raw zone spellings. `ZONE_MAP` collapses 
them to 7 canonical values. The MongoDB and R pipelines use the same map so 
all 3 paradigms agree on the same zones downstream.

In [ ]:
ZONE_MAP = {
    'AIRPORT': 'Airport',     'Airport': 'Airport',
    'CENTRAL': 'Central',     'Central': 'Central',     'Ctr': 'Central',
    'EAST': 'East',           'East': 'East',
    'NORTH': 'North',         'North': 'North',         'north': 'North',
    'RiverSide': 'Riverside', 'Riverside': 'Riverside',
    'SOUTH': 'South',         'South': 'South',
    'WEST': 'West',           'West': 'West',
}

def canon(z):
    return ZONE_MAP.get(z, z) if isinstance(z, str) else z

DATE_COLS = {
    'orders':     ['order_created_at'],
    'deliveries': ['dispatch_time', 'delivery_completed_at'],
    'app_events': ['event_at'],
}

for tab, cfg in SCHEMA.items():
    for col in cfg.get('zone_cols', []):
        if col in frames[tab].columns:
            frames[tab][col] = frames[tab][col].map(canon)
for tab, cols in DATE_COLS.items():
    for col in cols:
        if col in frames[tab].columns:
            frames[tab][col] = pd.to_datetime(frames[tab][col], errors='coerce')
print('zones canonicalised and dates parsed')

## 5. Post-clean scorecard

Same scorecard against the cleaned tables. Distinct zone counts should drop 
sharply; rows, nulls, duplicate PKs and FK violations should be unchanged 
(the cleanup is deliberately non-destructive).

In [ ]:
post = scorecard(frames, SCHEMA, 'Post-clean scorecard')
diff = (pre - post).fillna(0).astype(int)
print('\nDifference (pre - post)')
print('-' * 60)
print(diff)

## 6. Analytical wide table

One row per order, joined with delivery, customer and complaint count. 
`failed` is the derived binary outcome the model later predicts.

In [ ]:
orders = frames['orders'].copy()
deliveries = frames['deliveries'].copy()
customers = frames['customers'].copy()
complaints = frames['complaints'].copy()

complaint_counts = (complaints.groupby('order_id')
                              .size().rename('complaint_count').reset_index())

wide = (orders
        .merge(deliveries, on='order_id', how='left')
        .merge(customers, on='customer_id', how='left', suffixes=('', '_cust'))
        .merge(complaint_counts, on='order_id', how='left'))
wide['complaint_count'] = wide['complaint_count'].fillna(0).astype(int)
wide['failed'] = (wide['delivery_status'] == 'Failed').astype(int)
print(f'wide table: {wide.shape[0]} rows x {wide.shape[1]} cols')
wide.head(3)

## 7. Descriptive statistics

Failure rate by service type, then by pickup zone. The contrast frames the 
headline finding the inferential tests later confirm.

In [ ]:
by_service = (wide.dropna(subset=['delivery_status'])
                  .groupby('service_type')
                  .agg(orders=('order_id','count'),
                       failures=('failed','sum'),
                       failure_rate=('failed','mean'))
                  .round(3))
print('Failure rate by service type'); print(by_service)

In [ ]:
by_zone = (wide.dropna(subset=['delivery_status'])
               .groupby('pickup_zone')
               .agg(orders=('order_id','count'),
                    failures=('failed','sum'),
                    failure_rate=('failed','mean'))
               .sort_values('failure_rate', ascending=False).round(3))
print('Failure rate by pickup zone'); print(by_zone)

## 8. scipy chi square tests

T1 (pickup zone vs delivery outcome) is the test I expect to reach 
significance; T2 (service type) acts as a negative control. The R analytics 
notebook reproduces both tests with `chisq.test` and the numbers agree to 
4 decimal places.

In [ ]:
t1_table = pd.crosstab(wide['pickup_zone'], wide['delivery_status'])
chi2, p, dof, _ = stats.chi2_contingency(t1_table)
print(f'T1 (pickup_zone vs delivery_status): chi2 = {chi2:.2f}, dof = {dof}, p = {p:.4f}')

t2_table = pd.crosstab(wide['service_type'], wide['delivery_status'])
chi2, p, dof, _ = stats.chi2_contingency(t2_table)
print(f'T2 (service_type vs delivery_status): chi2 = {chi2:.2f}, dof = {dof}, p = {p:.4f}')

## 9. scikit-learn random forest

Probes whether delivery failure is predictable from order- and customer- 
level features. Stratified 5-fold cross-validation, ROC AUC as the headline 
metric. If both this model and the R logistic regression find nothing, the 
signal is not at order level.

In [ ]:
model_df = wide.dropna(subset=['delivery_status', 'route_distance_km']).copy()
categorical = ['service_type', 'priority_level', 'pickup_zone', 'customer_type']
numeric = ['order_value', 'promised_window_hours', 'route_distance_km',
           'loyalty_score', 'app_engagement_score']
for col in categorical + numeric:
    model_df = model_df.dropna(subset=[col])

X = model_df[categorical + numeric]
y = (model_df['delivery_status'] == 'Failed').astype(int)
print(f'sample: {len(y)} orders, failure rate {y.mean():.3f}')

pre_t = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', StandardScaler(), numeric),
])
pipe = Pipeline([('pre', pre_t),
                 ('rf', RandomForestClassifier(n_estimators=300, random_state=42))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'\n5-fold ROC AUC: {auc_scores.mean():.3f} +/- {auc_scores.std():.3f}')
print(f'per fold: {[round(s, 3) for s in auc_scores]}')

In [ ]:
# Hold-out evaluation: ROC + confusion matrix.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
pipe.fit(X_train, y_train)
probs = pipe.predict_proba(X_test)[:, 1]
preds = pipe.predict(X_test)
fpr, tpr, _ = roc_curve(y_test, probs)
test_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f'AUC = {test_auc:.3f}')
axes[0].plot([0, 1], [0, 1], '--', color='grey')
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC, hold-out test fold'); axes[0].legend()

cm = confusion_matrix(y_test, preds)
axes[1].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i, j], ha='center', va='center',
                     color='black' if cm[i, j] < cm.max()/2 else 'white')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(['Not failed', 'Failed'])
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(['Not failed', 'Failed'])
axes[1].set_title('Confusion matrix')
plt.tight_layout(); plt.show()

print(classification_report(y_test, preds, target_names=['Not failed', 'Failed']))

**Interpretation.** 5-fold ROC AUC sits around 0.49, indistinguishable from 
random guessing. The confusion matrix shows the model defaults to predicting 
non-failure for almost every order. The predictive signal in this dataset is 
at the zone level, not the order level: the same conclusion the R logistic 
regression reaches.

## 10. Rubric coverage

In [ ]:
rubric = pd.DataFrame([
    ['2. Extract',                  'Load 9 CSVs from the repo',         'Data acquisition'],
    ['3. Pre-clean scorecard',      '5-dimension quality survey',         'Data quality validation'],
    ['4. Transform',                'Zone canon + date parsing',          'Data cleaning'],
    ['5. Post-clean scorecard',     'Diff vs pre-clean baseline',         'Data quality validation'],
    ['6. Wide table',               'Joins orders + deliveries + cust.',  'Data engineering'],
    ['7. Descriptive stats',        'Failure rate by zone and service',   'Exploratory analysis'],
    ['8. scipy chi square',         'Inferential cross-check vs R',       'Statistical inference'],
    ['9. Random forest',            'Supervised ML with CV',              'Machine learning'],
], columns=['Section', 'What it shows', 'Rubric line (Python Data Processing, 20 marks)'])
print(rubric.to_string(index=False))